# 08.7 - Tokenizers (BPE, WordPiece)

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

Tokenizers convert raw text into numerical tokens that models can process. Subword tokenization methods like BPE and WordPiece split rare words into meaningful pieces while keeping common words as single tokens.

## 2. Why Does This Matter?

Tokenization is the bridge between raw text and model input. A bad tokenizer causes vocabulary mismatches, poor generation, and wasted capacity. Every transformer model depends on its tokenizer.

## 3. Prerequisites

- String processing, vocabulary concepts, basic probability.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain character-level vs word-level vs subword tokenization.
- Implement a minimal BPE tokenizer from scratch.
- Understand WordPiece's likelihood-based merge criterion.
- Use tokenizers correctly with different models.

## 5. Mental Model

A tokenizer is like a smart compression algorithm for text. It keeps frequent words whole ("the", "and") and breaks rare words into pieces ("transform" + "##er" + "##s"). BPE builds this by iteratively merging the most common character pairs.

```text
Raw text: "Tokenization is fascinating"
BPE tokens: ["Token", "ization", " is", " fasc", "inating"]
WordPiece tokens: ["Token", "ization", " is", "fasc", "##inating"]
```


## 6. Character vs Word vs Subword

Compare the three levels of tokenization on a small corpus.


In [1]:
import matplotlib
matplotlib.use('Agg')
import re
from collections import Counter

text = "Tokenization is fascinating and tokenization is powerful"

# Character-level
char_tokens = list(text)
print("Character-level tokens:", char_tokens[:20], "...")

# Word-level (simple whitespace split)
word_tokens = text.split()
print("Word-level tokens:", word_tokens)

# Subword-level (illustrative)
subword_tokens = ["Token", "ization", " is", " fasc", "inating", " and", " powerful"]
print("Subword-level tokens:", subword_tokens)

print("\nCharacter vocab is small but long sequences; word vocab is large but short sequences; subword balances both.")


Character-level tokens: ['T', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n', ' ', 'i', 's', ' ', 'f', 'a', 's', 'c'] ...
Word-level tokens: ['Tokenization', 'is', 'fascinating', 'and', 'tokenization', 'is', 'powerful']
Subword-level tokens: ['Token', 'ization', ' is', ' fasc', 'inating', ' and', ' powerful']

Character vocab is small but long sequences; word vocab is large but short sequences; subword balances both.


## 7. Implement a Minimal BPE Tokenizer from Scratch

BPE training: start with characters, iteratively merge the most frequent adjacent pair.


In [2]:
def get_stats(corpus):
    """Count adjacent character pairs across all words."""
    pairs = Counter()
    for word, freq in corpus.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge(corpus, pair, new_token):
    """Replace the most frequent pair with a new token."""
    out = {}
    bigram = ' '.join(pair)
    for word, freq in corpus.items():
        out[word.replace(bigram, new_token)] = freq
    return out

# Build a tiny corpus: word -> frequency
corpus = {
    "l o w </w>": 5,
    "l o w e r </w>": 2,
    "n e w e s t </w>": 6,
    "w i d e s t </w>": 3,
}

vocab_size = 20
merges = []
for i in range(vocab_size):
    pairs = get_stats(corpus)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    new_token = ''.join(best)
    corpus = merge(corpus, best, new_token)
    merges.append((best, new_token, pairs[best]))
    print(f"Merge {i+1}: {best} -> '{new_token}' (count={pairs[best]})")

print("\nFinal corpus tokens:")
for w, f in corpus.items():
    print(f"  {w} x{f}")


Merge 1: ('e', 's') -> 'es' (count=9)
Merge 2: ('es', 't') -> 'est' (count=9)
Merge 3: ('est', '</w>') -> 'est</w>' (count=9)
Merge 4: ('l', 'o') -> 'lo' (count=7)
Merge 5: ('lo', 'w') -> 'low' (count=7)
Merge 6: ('n', 'e') -> 'ne' (count=6)
Merge 7: ('ne', 'w') -> 'new' (count=6)
Merge 8: ('new', 'est</w>') -> 'newest</w>' (count=6)
Merge 9: ('low', '</w>') -> 'low</w>' (count=5)
Merge 10: ('w', 'i') -> 'wi' (count=3)
Merge 11: ('wi', 'd') -> 'wid' (count=3)
Merge 12: ('wid', 'est</w>') -> 'widest</w>' (count=3)
Merge 13: ('low', 'e') -> 'lowe' (count=2)
Merge 14: ('lowe', 'r') -> 'lower' (count=2)
Merge 15: ('lower', '</w>') -> 'lower</w>' (count=2)

Final corpus tokens:
  low</w> x5
  lower</w> x2
  newest</w> x6
  widest</w> x3


## 8. Encode and Decode with the Learned BPE

Use the learned merges to tokenize new text.


In [3]:
def encode(text, merges):
    """Apply learned merges to tokenize a word."""
    tokens = list(text) + ['</w>']
    while len(tokens) > 1:
        pairs = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
        # Find the first pair that is in our merge list
        merged = False
        for pair in pairs:
            for (a, b), new_tok, _ in merges:
                if pair == (a, b):
                    # merge this pair
                    idx = pairs.index(pair)
                    tokens = tokens[:idx] + [new_tok] + tokens[idx+2:]
                    merged = True
                    break
            if merged:
                break
        if not merged:
            break
    return tokens

for w in ["low", "lowest", "newer", "wider"]:
    print(f"{w:8s} -> {encode(w, merges)}")


low      -> ['low</w>']
lowest   -> ['lowe', 's', 't', '</w>']
newer    -> ['new', 'e', 'r', '</w>']
wider    -> ['wid', 'e', 'r', '</w>']


## 9. WordPiece vs BPE

WordPiece uses a likelihood-based merge criterion instead of raw frequency. The key difference is the scoring function.


In [4]:
# WordPiece scoring: score = (freq of pair) / (freq of first * freq of second)
# This prefers pairs that are frequent together relative to their individual frequencies.
def wordpiece_score(pair_freq, first_freq, second_freq):
    return pair_freq / (first_freq * second_freq)

# Example comparison
pairs = {
    ('l','o'): 5,   # frequent pair
    ('x','y'): 5,   # same raw frequency
}
first_freq = {'l': 7, 'x': 100}
second_freq = {'o': 7, 'y': 100}

for pair, f in pairs.items():
    s = wordpiece_score(f, first_freq[pair[0]], second_freq[pair[1]])
    print(f"Pair {pair}: raw freq={f}, WordPiece score={s:.4f}")

print("\nBPE picks by raw frequency; WordPiece normalizes by individual frequencies, favoring pairs that are relatively more cohesive.")


Pair ('l', 'o'): raw freq=5, WordPiece score=0.1020
Pair ('x', 'y'): raw freq=5, WordPiece score=0.0005

BPE picks by raw frequency; WordPiece normalizes by individual frequencies, favoring pairs that are relatively more cohesive.


## 10. Special Tokens

BERT uses [CLS], [SEP], [PAD], [MASK]; GPT uses <s>, </s>, <pad>. These special tokens affect model behavior.


In [5]:
special_tokens = {
    "BERT": ["[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    "GPT-2": ["<|endoftext|>"],
    "T5": ["<pad>", "</s>", "<unk>", "<s>"],
}
for model, toks in special_tokens.items():
    print(f"{model}: {toks}")

# Demonstrate why special tokens matter: they are reserved IDs in the vocab
print("\nSpecial tokens are reserved IDs that the model treats specially (e.g. [CLS] aggregates the whole sequence).")


BERT: ['[CLS]', '[SEP]', '[PAD]', '[MASK]']
GPT-2: ['<|endoftext|>']
T5: ['<pad>', '</s>', '<unk>', '<s>']

Special tokens are reserved IDs that the model treats specially (e.g. [CLS] aggregates the whole sequence).


## 11. Failure Case: Wrong Tokenizer

Using the wrong tokenizer for a model produces garbage.


In [6]:
# Illustrative: a tokenizer trained on English fails on non-English text
english_corpus = "the cat sat on the mat the dog ran fast"
vocab = set(english_corpus.split())
test_text = "le chat est assis sur le tapis"
unknown = [w for w in test_text.split() if w not in vocab]
print("English vocab:", sorted(vocab))
print("French text:", test_text)
print("Out-of-vocabulary words:", unknown)
print("\nThese would become [UNK] tokens, losing information. Subword tokenizers handle this better.")


English vocab: ['cat', 'dog', 'fast', 'mat', 'on', 'ran', 'sat', 'the']
French text: le chat est assis sur le tapis
Out-of-vocabulary words: ['le', 'chat', 'est', 'assis', 'sur', 'le', 'tapis']

These would become [UNK] tokens, losing information. Subword tokenizers handle this better.


## 12. Debugging: Common Errors

- **Wrong tokenizer**: model outputs garbage. Use the tokenizer matching the model.
- **Unknown tokens**: input outside vocabulary. Use BPE/SentencePiece for open vocab.
- **Token count != word count**: different tokenizers merge differently.
- **Poor non-English performance**: tokenizer trained on English only.

## 13. Real-World Considerations

- API pricing is per token, so tokenization affects cost.
- Save and version your tokenizer alongside your model.
- Test on edge cases: emoji, numbers, URLs, non-English text.

## 14. Common Mistakes

- Assuming token count = word count.
- Not handling special tokens correctly.
- Not tokenizing input the same way during training and inference.

## 15. When NOT to Use

- Word-level tokenization for open-vocabulary tasks.
- A tokenizer trained on a different domain than your data.

## 16. Challenge

Extend the BPE implementation to handle a larger corpus and compare vocabulary sizes.


In [7]:
# Challenge: tokenize a larger corpus and count unique tokens
big_corpus = {
    "t h e </w>": 10,
    "c a t </w>": 8,
    "c a t s </w>": 4,
    "d o g </w>": 7,
    "d o g s </w>": 3,
    "r u n </w>": 6,
    "r u n n i n g </w>": 2,
}
c = big_corpus.copy()
m = []
for i in range(15):
    pairs = get_stats(c)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    nt = ''.join(best)
    c = merge(c, best, nt)
    m.append((best, nt))

unique_tokens = set()
for w in c:
    unique_tokens.update(w.split())
print(f"After {len(m)} merges, unique subword tokens: {len(unique_tokens)}")
print("Learned merges:", [nt for _, nt in m])


After 15 merges, unique subword tokens: 11
Learned merges: ['ca', 'cat', 'th', 'the', 'the</w>', 'do', 'dog', 'cat</w>', 'ru', 'run', 's</w>', 'dog</w>', 'run</w>', 'cats</w>', 'dogs</w>']


## 17. Closed-Book Recall

Without looking back:

1. Why is subword tokenization preferred over word-level?
2. How does BPE decide which pairs to merge?
3. What special tokens does BERT need vs GPT?
4. How does vocabulary size affect model performance and cost?

## 18. Teach-Back Questions

Explain to another person:

- How BPE builds a vocabulary from a corpus.
- The difference between BPE and WordPiece merge criteria.

## 19. Summary

You implemented a minimal BPE tokenizer from scratch, compared character/word/subword levels, and understood WordPiece's likelihood-based merging.

## 20. Further Experiment

- Implement WordPiece scoring in the from-scratch tokenizer.
- Test your tokenizer on emoji, URLs, and code.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
